# Building and Populating a Data Product on AWS

## Instructions

In this exercise, you will be building and populating a database in AWS. You will then perform some CRUD queries to verify the functionallity of the database









```mermaid
erDiagram

    clients {
        varchar client_id PK
        varchar client_name
        varchar industry
        char state
        varchar client_status
        varchar sales_rep_id FK
    }

    sales_rep {
        varchar sales_rep_id PK
        varchar rep_first_name
        varchar rep_last_name
        varchar rep_email
        varchar territory
    }

    contact_log {
        varchar contact_log_id PK
        date contact_date
        varchar contact_method
        varchar contact_topic
        varchar contact_outcome
        numeric estimated_opportunity_value
        date next_follow_up_date
        varchar client_id FK
        varchar sales_rep_id FK
    }

    sales_rep ||--o{ clients : manages
    clients ||--o{ contact_log : has
    sales_rep ||--o{ contact_log : records

## Connect to AWS


In [ ]:
import boto3
import json
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values

AWS_ACCESS_KEY_ID = "<get key from classroom>"
AWS_SECRET_ACCESS_KEY = "<get key from classroom>"
AWS_SESSION_TOKEN = "<get key from classroom>"
AWS_REGION = "us-east-1"

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION,
)


# Get DB connection info from CloudFormation
cf = session.client("cloudformation")
stack = cf.describe_stacks()["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"]}

db_host = outputs["DBEndpoint"]
db_port = int(outputs["DBPort"])
db_name = outputs["DBName"]
secret_arn = outputs["MasterSecretArn"]

# Get credentials from Secrets Manager
sm = session.client("secretsmanager", region_name="us-east-1")
secret_response = sm.get_secret_value(SecretId=secret_arn)
db_secret = json.loads(secret_response["SecretString"])


db_user = db_secret["username"]
db_password = db_secret["password"]

# Connect to DB
try:
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_secret["username"],
        password=db_secret["password"],
        sslmode="require",
    )
    print("Successfully connected to the database")
 
except Exception as e:
    print(f"Connection failed: {type(e).__name__}: {e}")

### Step 1
Import the following CSV file into Python: client_contact_landing_100_rows.csv


---

In [ ]:
import pandas as pd
from io import StringIO
import psycopg2


CSV_FILE = "../build-populate-db-starter/client_contact_landing_100_rows.csv"


df = pd.read_csv(CSV_FILE)

print("CSV rows:", len(df))
display(df.head())

### Step 2 

Create Landing Table Capable of Holding Data 

Load Data into table

---

In [ ]:
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode="require"
)


create_landing_table_sql = """
DROP TABLE IF EXISTS client_contact_landing;

CREATE TABLE client_contact_landing (
    contact_log_id VARCHAR(20),
    contact_date DATE,
    contact_method VARCHAR(50),
    contact_topic VARCHAR(100),
    contact_outcome VARCHAR(100),
    estimated_opportunity_value NUMERIC(12,2),
    next_follow_up_date DATE,
    client_id VARCHAR(20),
    client_name VARCHAR(150),
    industry VARCHAR(100),
    state CHAR(2),
    client_status VARCHAR(50),
    sales_rep_id VARCHAR(20),
    rep_first_name VARCHAR(50),
    rep_last_name VARCHAR(50),
    rep_email VARCHAR(100),
    territory VARCHAR(50)
);
"""

try:
    with conn:
        with conn.cursor() as cur:
            # Create the landing table
            cur.execute(create_landing_table_sql)

            # Load the DataFrame into PostgreSQL using COPY
            csv_buffer = StringIO()
            df.to_csv(csv_buffer, index=False, header=False)
            csv_buffer.seek(0)

            copy_sql = """
                COPY client_contact_landing (
                    contact_log_id,
                    contact_date,
                    contact_method,
                    contact_topic,
                    contact_outcome,
                    estimated_opportunity_value,
                    next_follow_up_date,
                    client_id,
                    client_name,
                    industry,
                    state,
                    client_status,
                    sales_rep_id,
                    rep_first_name,
                    rep_last_name,
                    rep_email,
                    territory
                )
                FROM STDIN
                WITH (FORMAT CSV);
            """

            cur.copy_expert(copy_sql, csv_buffer)

    print("Landing table created and CSV data loaded successfully.")

finally:
    conn.close()

### Step 3
Using the Mermaid Diagram found above as your guide, create 3 normalized tables

---


In [ ]:
# Create normalized tables from the client_contact_landing table
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode="require"
)

cur = conn.cursor()

create_tables_sql = """
DROP TABLE IF EXISTS contact_log;
DROP TABLE IF EXISTS clients;
DROP TABLE IF EXISTS sales_rep;

CREATE TABLE sales_rep (
    sales_rep_id VARCHAR(20) PRIMARY KEY,
    rep_first_name VARCHAR(50) NOT NULL,
    rep_last_name VARCHAR(50) NOT NULL,
    rep_email VARCHAR(100) NOT NULL,
    territory VARCHAR(50) NOT NULL
);

CREATE TABLE clients (
    client_id VARCHAR(20) PRIMARY KEY,
    client_name VARCHAR(150) NOT NULL,
    industry VARCHAR(100) NOT NULL,
    state CHAR(2) NOT NULL,
    client_status VARCHAR(50) NOT NULL,
    sales_rep_id VARCHAR(20) NOT NULL,
    CONSTRAINT fk_clients_sales_rep
        FOREIGN KEY (sales_rep_id)
        REFERENCES sales_rep(sales_rep_id)
);

CREATE TABLE contact_log (
    contact_log_id VARCHAR(20) PRIMARY KEY,
    contact_date DATE NOT NULL,
    contact_method VARCHAR(50) NOT NULL,
    contact_topic VARCHAR(100) NOT NULL,
    contact_outcome VARCHAR(100) NOT NULL,
    estimated_opportunity_value NUMERIC(12,2),
    next_follow_up_date DATE,
    client_id VARCHAR(20) NOT NULL,
    sales_rep_id VARCHAR(20) NOT NULL,
    CONSTRAINT fk_contact_log_clients
        FOREIGN KEY (client_id)
        REFERENCES clients(client_id),
    CONSTRAINT fk_contact_log_sales_rep
        FOREIGN KEY (sales_rep_id)
        REFERENCES sales_rep(sales_rep_id)
);
"""

cur.execute(create_tables_sql)
conn.commit()

print("Tables created")


### Step 4
Populate the normalized tables with data from the landing table you created

---

In [ ]:
cur = conn.cursor()

sql_statements = [
    (
        "sales_rep",
        """
        INSERT INTO sales_rep
        SELECT DISTINCT
            sales_rep_id,
            rep_first_name,
            rep_last_name,
            rep_email,
            territory
        FROM client_contact_landing;
        """
    ),
    (
        "clients",
        """
        INSERT INTO clients
        SELECT DISTINCT
            client_id,
            client_name,
            industry,
            state,
            client_status,
            sales_rep_id
        FROM client_contact_landing;
        """
    ),
    (
        "contact_log",
        """
        INSERT INTO contact_log
        SELECT
            contact_log_id,
            contact_date,
            contact_method,
            contact_topic,
            contact_outcome,
            estimated_opportunity_value,
            next_follow_up_date,
            client_id,
            sales_rep_id
        FROM client_contact_landing;
        """
    )
]

for table_name, sql in sql_statements:
    cur.execute(sql)
    print(f"{table_name} table populated successfully.")

conn.commit()
cur.close()

In [ ]:
# Validate row counts

cur = conn.cursor()

row_count_sql = """
SELECT 'client_contact_landing' AS table_name, COUNT(*) AS row_count FROM client_contact_landing
UNION ALL
SELECT 'sales_rep' AS table_name, COUNT(*) AS row_count FROM sales_rep
UNION ALL
SELECT 'clients' AS table_name, COUNT(*) AS row_count FROM clients
UNION ALL
SELECT 'contact_log' AS table_name, COUNT(*) AS row_count FROM contact_log;
"""

cur.execute(row_count_sql)
results = cur.fetchall()

for row in results:
    print(row)

cur.close()

### Step 5
Perform 3 CRUD queries:

One Insert

One Delete

One Select query that joins all 3 tables

Make sure to run Select queries after the Insert and Delete queries to ensure they work properly

---

#### Insert

Insert a record into the database

In [ ]:
# CRUD Query 1: INSERT a new contact_log record

conn.rollback()  # clears any prior failed transaction state
cur = conn.cursor()

insert_sql = """
INSERT INTO contact_log (
    contact_log_id,
    contact_date,
    contact_method,
    contact_topic,
    contact_outcome,
    estimated_opportunity_value,
    next_follow_up_date,
    client_id,
    sales_rep_id
)
VALUES (
    'LOG99999',
    CURRENT_DATE,
    'Phone',
    'New Product Discussion',
    'Follow-up Needed',
    25000.00,
    CURRENT_DATE + INTERVAL '14 days',
    'CL0001',
    'SR001'
);
"""

cur.execute(insert_sql)
conn.commit()
cur.close()

### Verify the new record was inserted correctly
df_in = pd.read_sql("""
SELECT *
FROM contact_log
WHERE contact_log_id = 'LOG99999';
""", conn)

df_in

#### Delete 

Delete a record from the database

In [ ]:
# CRUD Query 2: DELETE the inserted contact_log record

conn.rollback()
cur = conn.cursor()

delete_sql = """
DELETE FROM contact_log
WHERE contact_log_id = 'LOG99999';
"""

cur.execute(delete_sql)
conn.commit()
cur.close()

print("Delete completed.")

# Verify DELETE worked

df_del = pd.read_sql("""
SELECT *
FROM contact_log
WHERE contact_log_id = 'LOG99999';
""", conn)

df_del

#### Join

Write a SELECT query that joins all 3 tables

In [ ]:
# CRUD Query 3: READ query joining all three tables

conn.rollback()
cur = conn.cursor()

join_sql = """
SELECT
    cl.contact_log_id,
    cl.contact_date,
    c.client_id,
    c.client_name,
    c.industry,
    c.client_status,
    sr.sales_rep_id,
    sr.rep_first_name || ' ' || sr.rep_last_name AS sales_rep_name,
    sr.territory,
    cl.contact_method,
    cl.contact_topic,
    cl.contact_outcome,
    cl.estimated_opportunity_value
FROM contact_log cl
INNER JOIN clients c
    ON cl.client_id = c.client_id
INNER JOIN sales_rep sr
    ON cl.sales_rep_id = sr.sales_rep_id
ORDER BY cl.contact_date DESC
LIMIT 20;
"""

cur.execute(join_sql)
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()